In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
import folium
from folium.plugins import PolyLineTextPath

from branca.element import Element, Template
import json

In [32]:
## TEST VERSION 2 (CHANGE OF THE ICONS IS DYNAMIC, BUT IS BUILT BASED ON EACH SINGLE SEGMENT OF ROUTE)
def plot_map(cmp_segid, segments, cmp_shp, xd_shp):
    seg = segments.loc[segments['cmp_segid'].eq(cmp_segid)]
    xd_shp = xd_shp.loc[xd_shp['XDSegID'].isin(seg['inrix_segid'])].to_crs('epsg:4326')

    xd_shp = pd.merge(xd_shp, seg[['inrix_segid','old','new','length_matched_new']],
                      left_on='XDSegID', right_on='inrix_segid')
    
    cmp_shp = cmp_shp.loc[cmp_shp['cmp_segid'].eq(cmp_segid)].to_crs('epsg:4326')
    
    
    
    if seg.empty:
        print(f"No matched_path_gdf features for trip {trip_id_to_plot}")
        return folium.Map()  # empty base map
    
    # -- Create map --
    center = xd_shp.geometry.unary_union.centroid.coords[0]
    m = folium.Map(location=[center[1], center[0]], zoom_start=15, tiles="cartodbpositron")
    
    for _, row in xd_shp.iterrows():
        color = "#007AFF"
        weight = 3
        if row['old'] == 0:
            color = "#32fbe0"
            weight = 5
        coords = [(pt[1], pt[0]) for pt in row.geometry.coords]
    
        # 1) Draw the base polyline
        # 1a) Add a popup to each segment showing its sequence (rownum) and OSMID
        popup = folium.Popup(
            f"XDSegID: {row['XDSegID']}<br>Length Matched: {row['length_matched_new']}",
            max_width=200, sticky = True
        )
        tooltip = folium.Tooltip(
            f"XDSegID: {row['XDSegID']}<br>Length Matched: {row['length_matched_new']}",
            sticky = True
        )
        popup.options.update({
            "autoClose": False,
            "closeOnClick": True
        })
        
        poly = folium.PolyLine(
            locations=coords,
            color=color,
            opacity=0.8,
            popup = popup,
            tooltip = tooltip,
            weight = weight,
        ).add_to(m)

        # 2) Add ▶ arrowheads (and "=") along the line, letting Leaflet.TextPath auto-rotate ▶ 
        arrow_layer = PolyLineTextPath(
            poly,
            text="=▶",
            repeat="50%",      # percent‐based spacing (initial)
            offset=15,          # pixels above the centerline
            orientation="auto",# auto-rotate the ▶ glyph along the segment
            attributes={
                "fill": "#969696",
                "font-size": "12px",
                "font-weight": "bold",
            }
        ).add_to(m)
    
    color = "#eb6b34"
    coords = []
    for geom in cmp_shp.iloc[0].geometry.geoms:
        for pt in geom.coords:
            coords.append((pt[1], pt[0]))
    cmp_poly = folium.PolyLine(
        locations=coords,
        color=color,
        opacity=0.8,
        #popup = popup
    ).add_to(m)


    # -- Display map --
    return m

In [4]:
OLD = r'Q:\CMP\LOS Monitoring 2022\Network_Conflation\v2202\conflation_script_test\CMP_Segment_INRIX_Links_Correspondence_2202_Manual.csv'
NEW = r'Q:\CMP\LOS Monitoring 2025\Network_Conflation\v2501\CMP_Segment_INRIX_Links_Correspondence_2501_Manual-expandednetwork.csv'

In [5]:
XD = 'Q:/GIS/Transportation/Roads/INRIX/XD/2501/INRIX_XD-SF-2501.gpkg'
CMP = r'Q:\GIS\Transportation\Roads\CMP\cmp_roadway_segments-expanded-v202204.gpkg'

In [6]:
old = pd.read_csv(OLD)
old.rename(columns={c:c.lower() for c in old.columns}, inplace=True)
old['length_matched'] = old['length_matched'].round(2)
new = pd.read_csv(NEW)
new.rename(columns={c:c.lower() for c in new.columns}, inplace=True)
new['length_matched'] = new['length_matched'].round(2)

In [7]:
xd = gpd.read_file(XD)

In [8]:
cmp = gpd.read_file(CMP)

In [9]:
old['old'] = 1
new['new'] = 1

In [10]:
df = pd.merge(old, 
              new, 
              on=['cmp_segid','inrix_segid'], 
              how='outer',
              suffixes=['_old','_new']).fillna(0)

In [11]:
df.loc[df['cmp_segid'].eq(10)]

,cmp_segid,inrix_segid,length_matched_old,old,length_matched_new,new
146,10,429475281,625.58,1.0,624.87,1.0
147,10,449826929,629.37,1.0,630.67,1.0
148,10,449826930,630.66,1.0,626.17,1.0
149,10,449826931,345.38,1.0,361.92,1.0
150,10,1626746166,88.43,1.0,98.54,1.0
151,10,170081991,225.84,1.0,226.23,1.0
152,10,170663170,579.74,1.0,580.69,1.0
4646,10,485562815,0.00,0.0,605.05,1.0
4647,10,485591185,0.00,0.0,651.02,1.0
4648,10,485565285,0.00,0.0,628.03,1.0


In [12]:
added_xd_iter = iter(df.loc[df['old'].eq(0)].groupby('cmp_segid'))

In [33]:
cmp_segid = next(added_xd_iter)[0]
print(cmp_segid)
plot_map(cmp_segid, df, cmp, xd)

10
